# PCB 결함 검사 — 콜랩 Sanity Check

Team Convex | ATI 인턴 프로젝트

이 노트북은 새 콜랩 환경에서 repo 클론 → 의존성 설치 → 데이터 마운트 → **1-epoch 스모크 트레인** 까지
한 번에 검증하는 골격입니다.

> **실행 순서**: 셀을 위에서 아래로 순서대로 실행하세요.

## 셀 1 — Repo 클론 & 의존성 설치

In [ ]:
# TODO: 실제 GitHub repo URL 로 변경
REPO_URL = "https://github.com/YOUR_ORG/pcb-project.git"  # TODO

import os
if not os.path.exists("/content/pcb-project"):
    !git clone {REPO_URL} /content/pcb-project
%cd /content/pcb-project

!pip install -q -r requirements.txt
print("설치 완료")

## 셀 2 — 데이터 준비

콜랩에서는 Google Drive 마운트 또는 직접 다운로드로 DeepPCB 를 `/content/DeepPCB` 에 준비합니다.

In [ ]:
# TODO: 방법 A — Google Drive 마운트 (팀 공유 드라이브 사용 시)
# from google.colab import drive
# drive.mount('/content/drive')
# !ln -s /content/drive/MyDrive/DeepPCB /content/DeepPCB

# TODO: 방법 B — 직접 다운로드 (공개 URL 이 있는 경우)
# !wget -q DEEPPCB_URL -O /content/deeppcb.zip
# !unzip -q /content/deeppcb.zip -d /content/DeepPCB

import os
assert os.path.exists("/content/DeepPCB"), "DeepPCB 경로가 없습니다. 위 TODO 를 완료하세요."
print("데이터 경로 확인:", os.listdir("/content/DeepPCB")[:5])

## 셀 3 — config.yaml 콜랩 환경으로 전환 & 전처리

In [ ]:
import yaml

# env 를 colab 으로 전환
with open("config.yaml", "r") as f:
    cfg = yaml.safe_load(f)
cfg["env"] = "colab"
with open("config.yaml", "w") as f:
    yaml.dump(cfg, f)
print("env:", cfg["env"])

# 전처리 실행 (소규모 샘플로 빠르게 확인)
# TODO: 전체 데이터 대신 일부 샘플만 처리하는 --sample N 플래그 추가 고려
!python src/preprocess.py --config config.yaml

## 셀 4 — 스모크 트레인 (epochs=1)

In [ ]:
import yaml

# epochs 를 1로 임시 변경
with open("config.yaml", "r") as f:
    cfg = yaml.safe_load(f)
cfg["train"]["epochs"] = 1
with open("config.yaml", "w") as f:
    yaml.dump(cfg, f)

!python src/train.py --config config.yaml

## 셀 5 — 판정 레이어 동작 확인 (inspect)

In [ ]:
import sys
sys.path.insert(0, "src")

from ultralytics import YOLO
from utils import load_config, get_paths
from inspect import inspect_image

cfg = load_config("config.yaml")
paths = get_paths(cfg)

weight = paths["weights"] / "best.pt"
assert weight.exists(), f"best.pt 가 없습니다: {weight}"
model = YOLO(str(weight))

# TODO: 테스트할 샘플 이미지 경로로 변경
SAMPLE_IMAGE = str(paths["processed"] / "images" / "test" / "SAMPLE.jpg")  # TODO

result = inspect_image(SAMPLE_IMAGE, model, cfg)
print("판정:", result["verdict"])
print("결함 수:", result["defect_count"])
print("클래스별:", result["by_class"])
if result["review"]:
    print("REVIEW 대상:", result["review"])